## Live Yelp Review Extraction and Analysis

In [17]:
import pickle
import pandas as pd
import scipy.sparse as sp
import re
import time

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [18]:
with open("isolation_forest_yelp.pkl", "rb") as f:
    iso_model = pickle.load(f)

with open("tfidf_vectorizer.pkl", "rb") as f:
    tfidf = pickle.load(f)

print("Model and vectorizer loaded")

Model and vectorizer loaded


In [32]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

In [33]:
def preprocess_text(text):

    text = text.lower()

    text = re.sub(r"http\S+", "", text)

    text = re.sub(r"[^a-z\s]", "", text)

    tokens = text.split()

    cleaned_tokens = [
        lemmatizer.lemmatize(word)
        for word in tokens
        if word not in stop_words
    ]

    return " ".join(cleaned_tokens)

In [30]:
def create_driver():

    chrome_options = Options()

    chrome_options.add_argument("--start-maximized")

    chrome_options.add_argument(
        r"--user-data-dir=C:\Users\acer\AppData\Local\Google\Chrome\User Data"
    )

    chrome_options.add_argument("--profile-directory=Default")

    driver = webdriver.Chrome(options=chrome_options)

    return driver

In [43]:
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

def fetch_yelp_reviews(url):

    driver = create_driver()

    driver.get(url)

    wait = WebDriverWait(driver, 20)

    # wait until review elements appear
    try:
        wait.until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "span[lang='en']"))
        )
    except:
        print("Reviews did not load")
        driver.quit()
        return []

    reviews = []

    elements = driver.find_elements(By.CSS_SELECTOR, "span[lang='en']")

    for el in elements:

        text = el.text.strip()

        if len(text) > 40:
            reviews.append(text)

    driver.quit()

    return reviews[:20]

In [44]:
def analyze_yelp_business(url):

    reviews = fetch_yelp_reviews(url)

    print("Reviews collected:", len(reviews))

    if len(reviews) == 0:
        print("No reviews extracted")
        return

    clean_reviews = [preprocess_text(r) for r in reviews]

    X_text = tfidf.transform(clean_reviews)

    review_length = [[len(r.split())] for r in clean_reviews]

    X_live = sp.hstack([X_text, review_length])

    predictions = iso_model.predict(X_live)

    suspicious = [1 if p == -1 else 0 for p in predictions]

    suspicious_percentage = (sum(suspicious) / len(suspicious)) * 100

    print("\nSuspicious review percentage:", round(suspicious_percentage,2), "%")

    if suspicious_percentage > 20:
        print("⚠️ Business may have suspicious review activity")
    else:
        print("✅ Reviews appear mostly genuine")

In [45]:
url = "https://www.yelp.com/biz/four-barrel-coffee-san-francisco"

analyze_yelp_business(url)

SessionNotCreatedException: Message: session not created: Chrome failed to start: crashed.
  (session not created: DevToolsActivePort file doesn't exist)
  (The process started from chrome location C:\Program Files\Google\Chrome\Application\chrome.exe is no longer running, so ChromeDriver is assuming that Chrome has crashed.); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#sessionnotcreatedexception
Stacktrace:
Symbols not available. Dumping unresolved backtrace:
	0x7ff623ea4125
	0x7ff623ea4180
	0x7ff623c3d72d
	0x7ff623c7e547
	0x7ff623c795f9
	0x7ff623ccfc61
	0x7ff623ccf4c6
	0x7ff623c890b8
	0x7ff623c89fa3
	0x7ff62416fe10
	0x7ff62416a2bd
	0x7ff62418a54a
	0x7ff623ec0725
	0x7ff623ec919c
	0x7ff623ead564
	0x7ff623ead716
	0x7ff623e933f7
	0x7ffe5e1ee8d7
	0x7ffe5f7cc48c
